# Complex Compare Results Notebook

This notebook analyzes the complex `compare-models` runs stored in the `Complex_Commands` folder. It is intended to compare how different retrieval setups and LLMs handle multi-course reasoning questions.

The notebook focuses on:

- cross-course evaluation comparison
- cross-course topic comparison
- security-course comparison
- theory-heavy vs application-oriented comparison
- side-by-side model outputs for the same question


In [1]:
from pathlib import Path
import json
import re

import pandas as pd
from IPython.display import display

pd.set_option('display.max_colwidth', 300)

ROOT = Path('.')
COMPLEX_DIR = ROOT / 'Complex_Commands'


In [2]:
def short_embedding(model_name: str) -> str:
    model_name = (model_name or '').lower()
    if 'bge' in model_name:
        return 'BGE'
    if 'minilm' in model_name:
        return 'MiniLM'
    return model_name.split('/')[-1] or 'unknown'


def normalize_question(question: str) -> str:
    q = re.sub(r'\s+', ' ', (question or '').strip().lower())
    if 'compare the evaluation criteria of generative ai and llms, iot networks, and applied machine learning in health care' in q:
        return 'complex_eval_comparison'
    if 'compare the topics covered in generative ai and llms, making causal inferences, and applied machine learning in health care' in q:
        return 'complex_topics_comparison'
    if 'compare cryptography and network security' in q:
        return 'complex_security_comparison'
    if 'which courses appear more theory-heavy and which appear more application-oriented' in q:
        return 'complex_theory_vs_application'
    return q


QUESTION_LABELS = {
    'complex_eval_comparison': 'Evaluation comparison across courses',
    'complex_topics_comparison': 'Topics comparison across ML-related courses',
    'complex_security_comparison': 'Cryptography vs Network Security comparison',
    'complex_theory_vs_application': 'Theory-heavy vs application-oriented comparison',
}


def detect_config_from_file_name(file_name: str) -> tuple[str, str]:
    name = file_name.lower()
    vector_store = 'unknown'
    if 'faiss' in name:
        vector_store = 'FAISS'
    elif 'chroma' in name:
        vector_store = 'Chroma'

    embedding = 'unknown'
    if 'bge' in name:
        embedding = 'BGE'
    elif 'minilm' in name:
        embedding = 'MiniLM'

    return vector_store, embedding


def load_complex_compare_runs(folder: Path) -> pd.DataFrame:
    rows = []
    for path in sorted(folder.glob('*.json')):
        payload = json.loads(path.read_text(encoding='utf-8'))
        question = payload.get('question', '')
        question_key = normalize_question(question)
        question_label = QUESTION_LABELS.get(question_key, question)
        vector_store, embedding = detect_config_from_file_name(path.name)
        retrieved_context = payload.get('retrieved_context', [])
        retrieved_sources = sorted({item.get('source', 'unknown') for item in retrieved_context})

        for model_output in payload.get('model_outputs', []):
            rows.append({
                'file': path.name,
                'question': question,
                'question_key': question_key,
                'question_label': question_label,
                'vector_store': vector_store,
                'embedding': embedding,
                'config': f'{vector_store} + {embedding}',
                'model': model_output.get('model'),
                'answer': model_output.get('answer'),
                'sources': ', '.join(model_output.get('sources', [])),
                'retrieved_context_count': len(retrieved_context),
                'retrieved_sources': ', '.join(retrieved_sources),
            })
    return pd.DataFrame(rows)


In [3]:
complex_df = load_complex_compare_runs(COMPLEX_DIR)
print('Complex compare logs found:', len(complex_df))
display(complex_df[['question_label', 'config', 'model', 'file']].sort_values(['question_label', 'config', 'model']).reset_index(drop=True))


Complex compare logs found: 12


,question_label,config,model,file
0,Cryptography vs Network Security comparison,Chroma + BGE,meta-llama/Llama-3.1-8B-Instruct,compare_models_20260404_203826_chroma_baai_bge_base_en_v1_5_meta_llama_llama_3_1__compare_cryptography_and_network_security_in_terms.json
1,Cryptography vs Network Security comparison,Chroma + BGE,openai/gpt-oss-120b,compare_models_20260404_203826_chroma_baai_bge_base_en_v1_5_meta_llama_llama_3_1__compare_cryptography_and_network_security_in_terms.json
2,Evaluation comparison across courses,Chroma + BGE,meta-llama/Llama-3.1-8B-Instruct,compare_models_20260404_203933_chroma_baai_bge_base_en_v1_5_meta_llama_llama_3_1__compare_the_evaluation_criteria_of_generative_ai_a.json
3,Evaluation comparison across courses,Chroma + BGE,openai/gpt-oss-120b,compare_models_20260404_203933_chroma_baai_bge_base_en_v1_5_meta_llama_llama_3_1__compare_the_evaluation_criteria_of_generative_ai_a.json
4,Evaluation comparison across courses,FAISS + BGE,meta-llama/Llama-3.1-8B-Instruct,compare_models_20260404_202433_faiss_baai_bge_base_en_v1_5_meta_llama_llama_3_1_8_compare_the_evaluation_criteria_of_generative_ai_a.json
5,Evaluation comparison across courses,FAISS + BGE,openai/gpt-oss-120b,compare_models_20260404_202433_faiss_baai_bge_base_en_v1_5_meta_llama_llama_3_1_8_compare_the_evaluation_criteria_of_generative_ai_a.json
6,Theory-heavy vs application-oriented comparison,Chroma + MiniLM,meta-llama/Llama-3.1-8B-Instruct,compare_models_20260404_204306_chroma_sentence_transformers_all_minilm_l6_v2_meta_across_all_uploaded_handouts_which_courses_appear_.json
7,Theory-heavy vs application-oriented comparison,Chroma + MiniLM,openai/gpt-oss-120b,compare_models_20260404_204306_chroma_sentence_transformers_all_minilm_l6_v2_meta_across_all_uploaded_handouts_which_courses_appear_.json
8,Topics comparison across ML-related courses,Chroma + BGE,meta-llama/Llama-3.1-8B-Instruct,compare_models_20260404_203729_chroma_baai_bge_base_en_v1_5_meta_llama_llama_3_1__compare_the_topics_covered_in_generative_ai_and_ll.json
9,Topics comparison across ML-related courses,Chroma + BGE,meta-llama/Llama-3.1-8B-Instruct,compare_models_20260404_204221_chroma_baai_bge_base_en_v1_5_meta_llama_llama_3_1__compare_the_topics_covered_in_generative_ai_and_ll.json


In [4]:
summary_df = complex_df[['question_label', 'config', 'model', 'retrieved_context_count', 'retrieved_sources']].drop_duplicates().sort_values(['question_label', 'config', 'model'])
display(summary_df.reset_index(drop=True))


,question_label,config,model,retrieved_context_count,retrieved_sources
0,Cryptography vs Network Security comparison,Chroma + BGE,meta-llama/Llama-3.1-8B-Instruct,0,
1,Cryptography vs Network Security comparison,Chroma + BGE,openai/gpt-oss-120b,0,
2,Evaluation comparison across courses,Chroma + BGE,meta-llama/Llama-3.1-8B-Instruct,0,
3,Evaluation comparison across courses,Chroma + BGE,openai/gpt-oss-120b,0,
4,Evaluation comparison across courses,FAISS + BGE,meta-llama/Llama-3.1-8B-Instruct,8,"2025_course_handout_AMCH.pdf, Generative AI and LLMs_CourseHandout.pdf"
5,Evaluation comparison across courses,FAISS + BGE,openai/gpt-oss-120b,8,"2025_course_handout_AMCH.pdf, Generative AI and LLMs_CourseHandout.pdf"
6,Theory-heavy vs application-oriented comparison,Chroma + MiniLM,meta-llama/Llama-3.1-8B-Instruct,10,"IoT Networks, Architectures and Applications-CSE 2023 Batch.pdf, 2025_course_handout_AMCH.pdf, Course Handout - ToC (Jan-June).pdf, Cryptography_CSE3703.pdf, Generative AI and LLMs_CourseHandout.pdf"
7,Theory-heavy vs application-oriented comparison,Chroma + MiniLM,openai/gpt-oss-120b,10,"IoT Networks, Architectures and Applications-CSE 2023 Batch.pdf, 2025_course_handout_AMCH.pdf, Course Handout - ToC (Jan-June).pdf, Cryptography_CSE3703.pdf, Generative AI and LLMs_CourseHandout.pdf"
8,Topics comparison across ML-related courses,Chroma + BGE,meta-llama/Llama-3.1-8B-Instruct,0,
9,Topics comparison across ML-related courses,Chroma + BGE,openai/gpt-oss-120b,0,


In [5]:
pivot_answers = complex_df.pivot_table(index=['question_label', 'config'], columns='model', values='answer', aggfunc='first')
display(pivot_answers)


model                                                                                                                                                                                                                                                                                                                                       meta-llama/Llama-3.1-8B-Instruct  \
question_label                                  config                                                                                                                                                                                                                                                                                                                         
Cryptography vs Network Security comparison     Chroma + BGE     Cryptography and Network Security courses focus on different aspects of information security, with Cryptography primarily focusing on the mathematical and theoretical aspects of secure data transmission, while Network Security emphasizes the practical implementation of security measures in compu...   
Evaluation comparison across courses            Chroma + BGE     The evaluation criteria for Generative AI, LLMs, IoT Networks, and Applied Machine Learning in Health Care are as follows: Generative AI is evaluated based on 30% quizzes, 20% assignments, 20% projects, 15% practical work, and 15% end-sem evaluation, while LLMs is evaluated based on 25% quizzes,...   
                                                FAISS + BGE      The evaluation criteria of Generative AI and LLMs, IoT Networks, and Applied Machine Learning in Health Care differ in terms of weightage of quizzes, assignments, projects, practical work, and end-sem evaluation. The Generative AI and LLMs course has a 20% weightage for quizzes, 40% for assignme...   
Theory-heavy vs application-oriented comparison Chroma + MiniLM  Courses that appear more theory-heavy are Theory of Computation (CSE2721) and Cryptography (CSE3703), while courses that appear more application-oriented are Generative AI and LLMs (CSE3720) and IoT Networks, Architectures and Applications (CSE3722). Theory of Computation (CSE2721) is a theory-h...   
Topics comparison across ML-related courses     Chroma + BGE     The topics covered in Generative AI and LLMs, Making Causal Inferences, and Applied Machine Learning in Health Care overlap in their focus on machine learning techniques, but each course has a unique focus. Generative AI and LLMs focuses on the development and application of large language model...   

model                                                                                                                                                                                                                                                                                                                                                    openai/gpt-oss-120b  
question_label                                  config                                                                                                                                                                                                                                                                                                                        
Cryptography vs Network Security comparison     Chroma + BGE            The information is not available in the uploaded handouts. The retrieved context does not contain any details about the Cryptography or Network Security courses, such as their focus, topics, or practical components. Since no handout files were provided, no source filenames can be referenced.  
Evaluation comparison across courses            Chroma + BGE     The evaluation criteria for Generative AI and LLMs, IoT Networks, and Applied Machine Learning in Health Care are not provided in the uploaded handouts. The retrieved context contains no details on quiz, assignment, project, practical work, or end‑semester 

In [6]:
for question_label in sorted(complex_df['question_label'].unique()):
    print('=' * 120)
    print(question_label)
    display(
        complex_df[complex_df['question_label'] == question_label][['config', 'model', 'answer', 'sources', 'retrieved_sources']]
        .sort_values(['config', 'model'])
        .reset_index(drop=True)
    )


Cryptography vs Network Security comparison


,config,model,answer,sources,retrieved_sources
0,Chroma + BGE,meta-llama/Llama-3.1-8B-Instruct,"Cryptography and Network Security courses focus on different aspects of information security, with Cryptography primarily focusing on the mathematical and theoretical aspects of secure data transmission, while Network Security emphasizes the practical implementation of security measures in compu...",,
1,Chroma + BGE,openai/gpt-oss-120b,"The information is not available in the uploaded handouts. The retrieved context does not contain any details about the Cryptography or Network Security courses, such as their focus, topics, or practical components. Since no handout files were provided, no source filenames can be referenced.",,


Evaluation comparison across courses


,config,model,answer,sources,retrieved_sources
0,Chroma + BGE,meta-llama/Llama-3.1-8B-Instruct,"The evaluation criteria for Generative AI, LLMs, IoT Networks, and Applied Machine Learning in Health Care are as follows: Generative AI is evaluated based on 30% quizzes, 20% assignments, 20% projects, 15% practical work, and 15% end-sem evaluation, while LLMs is evaluated based on 25% quizzes,...",,
1,Chroma + BGE,openai/gpt-oss-120b,"The evaluation criteria for Generative AI and LLMs, IoT Networks, and Applied Machine Learning in Health Care are not provided in the uploaded handouts. The retrieved context contains no details on quiz, assignment, project, practical work, or end‑semester weightages for these courses. Consequen...",,
2,FAISS + BGE,meta-llama/Llama-3.1-8B-Instruct,"The evaluation criteria of Generative AI and LLMs, IoT Networks, and Applied Machine Learning in Health Care differ in terms of weightage of quizzes, assignments, projects, practical work, and end-sem evaluation. The Generative AI and LLMs course has a 20% weightage for quizzes, 40% for assignme...","2025_course_handout_AMCH.pdf, Generative AI and LLMs_CourseHandout.pdf","2025_course_handout_AMCH.pdf, Generative AI and LLMs_CourseHandout.pdf"
3,FAISS + BGE,openai/gpt-oss-120b,"The Generative AI and LLMs course assigns 20 % to a quiz, 40 % to two programming assignments (each 20 %), and 40 % to the end‑semester project, while the Applied Machine Learning in Health Care course allocates 20 % to quizzes, 20 % to an assignment, 20 % to a mid‑term presentation, and 40 % to...","2025_course_handout_AMCH.pdf, Generative AI and LLMs_CourseHandout.pdf","2025_course_handout_AMCH.pdf, Generative AI and LLMs_CourseHandout.pdf"


Theory-heavy vs application-oriented comparison


,config,model,answer,sources,retrieved_sources
0,Chroma + MiniLM,meta-llama/Llama-3.1-8B-Instruct,"Courses that appear more theory-heavy are Theory of Computation (CSE2721) and Cryptography (CSE3703), while courses that appear more application-oriented are Generative AI and LLMs (CSE3720) and IoT Networks, Architectures and Applications (CSE3722). Theory of Computation (CSE2721) is a theory-h...","IoT Networks, Architectures and Applications-CSE 2023 Batch.pdf, 2025_course_handout_AMCH.pdf, Course Handout - ToC (Jan-June).pdf, Cryptography_CSE3703.pdf, Generative AI and LLMs_CourseHandout.pdf","IoT Networks, Architectures and Applications-CSE 2023 Batch.pdf, 2025_course_handout_AMCH.pdf, Course Handout - ToC (Jan-June).pdf, Cryptography_CSE3703.pdf, Generative AI and LLMs_CourseHandout.pdf"
1,Chroma + MiniLM,openai/gpt-oss-120b,"The Theory of Computation (CSE2721) and Cryptography (CSE3703) appear more theory‑heavy, while Generative AI and LLMs (CSE3720) and IoT Networks, Architectures and Applications (CSE3722) are more application‑oriented. The Theory of Computation handout provides only credit information and no prac...","IoT Networks, Architectures and Applications-CSE 2023 Batch.pdf, 2025_course_handout_AMCH.pdf, Course Handout - ToC (Jan-June).pdf, Cryptography_CSE3703.pdf, Generative AI and LLMs_CourseHandout.pdf","IoT Networks, Architectures and Applications-CSE 2023 Batch.pdf, 2025_course_handout_AMCH.pdf, Course Handout - ToC (Jan-June).pdf, Cryptography_CSE3703.pdf, Generative AI and LLMs_CourseHandout.pdf"


Topics comparison across ML-related courses


,config,model,answer,sources,retrieved_sources
0,Chroma + BGE,meta-llama/Llama-3.1-8B-Instruct,"The topics covered in Generative AI and LLMs, Making Causal Inferences, and Applied Machine Learning in Health Care overlap in their focus on machine learning techniques, but each course has a unique focus. Generative AI and LLMs focuses on the development and application of large language model...",,
1,Chroma + BGE,meta-llama/Llama-3.1-8B-Instruct,"The topics covered in Generative AI and LLMs, Making Causal Inferences, and Applied Machine Learning in Health Care have some overlap but distinct focuses. Generative AI and LLMs primarily focus on the development and application of large language models and generative AI techniques, with an emp...",,
2,Chroma + BGE,openai/gpt-oss-120b,"The information comparing the topics covered in Generative AI and LLMs, Making Causal Inferences, and Applied Machine Learning in Health Care is not available in the uploaded handouts. The retrieved context does not contain any details about the syllabus units, course outlines, or specific conte...",,
3,Chroma + BGE,openai/gpt-oss-120b,"The handouts do not provide enough detail to compare the topics covered in Generative AI and LLMs, Making Causal Inferences, and Applied Machine Learning in Health Care, nor to identify any overlap or unique focus for each course. The retrieved context contains no specific information about the ...",,


In [7]:
report_table = complex_df[['question_label', 'config', 'model', 'answer']].copy()
report_table['answer_preview'] = report_table['answer'].str.slice(0, 220)
display(report_table[['question_label', 'config', 'model', 'answer_preview']].sort_values(['question_label', 'config', 'model']).reset_index(drop=True))


,question_label,config,model,answer_preview
0,Cryptography vs Network Security comparison,Chroma + BGE,meta-llama/Llama-3.1-8B-Instruct,"Cryptography and Network Security courses focus on different aspects of information security, with Cryptography primarily focusing on the mathematical and theoretical aspects of secure data transmission, while Network Se"
1,Cryptography vs Network Security comparison,Chroma + BGE,openai/gpt-oss-120b,"The information is not available in the uploaded handouts. The retrieved context does not contain any details about the Cryptography or Network Security courses, such as their focus, topics, or practical components. Sinc"
2,Evaluation comparison across courses,Chroma + BGE,meta-llama/Llama-3.1-8B-Instruct,"The evaluation criteria for Generative AI, LLMs, IoT Networks, and Applied Machine Learning in Health Care are as follows: Generative AI is evaluated based on 30% quizzes, 20% assignments, 20% projects, 15% practical wor"
3,Evaluation comparison across courses,Chroma + BGE,openai/gpt-oss-120b,"The evaluation criteria for Generative AI and LLMs, IoT Networks, and Applied Machine Learning in Health Care are not provided in the uploaded handouts. The retrieved context contains no details on quiz, assignment, proj"
4,Evaluation comparison across courses,FAISS + BGE,meta-llama/Llama-3.1-8B-Instruct,"The evaluation criteria of Generative AI and LLMs, IoT Networks, and Applied Machine Learning in Health Care differ in terms of weightage of quizzes, assignments, projects, practical work, and end-sem evaluation. The Gen"
5,Evaluation comparison across courses,FAISS + BGE,openai/gpt-oss-120b,"The Generative AI and LLMs course assigns 20 % to a quiz, 40 % to two programming assignments (each 20 %), and 40 % to the end‑semester project, while the Applied Machine Learning in Health Care course allocates 20 % to"
6,Theory-heavy vs application-oriented comparison,Chroma + MiniLM,meta-llama/Llama-3.1-8B-Instruct,"Courses that appear more theory-heavy are Theory of Computation (CSE2721) and Cryptography (CSE3703), while courses that appear more application-oriented are Generative AI and LLMs (CSE3720) and IoT Networks, Architectur"
7,Theory-heavy vs application-oriented comparison,Chroma + MiniLM,openai/gpt-oss-120b,"The Theory of Computation (CSE2721) and Cryptography (CSE3703) appear more theory‑heavy, while Generative AI and LLMs (CSE3720) and IoT Networks, Architectures and Applications (CSE3722) are more application‑oriented. Th"
8,Topics comparison across ML-related courses,Chroma + BGE,meta-llama/Llama-3.1-8B-Instruct,"The topics covered in Generative AI and LLMs, Making Causal Inferences, and Applied Machine Learning in Health Care overlap in their focus on machine learning techniques, but each course has a unique focus. Generative AI"
9,Topics comparison across ML-related courses,Chroma + BGE,meta-llama/Llama-3.1-8B-Instruct,"The topics covered in Generative AI and LLMs, Making Causal Inferences, and Applied Machine Learning in Health Care have some overlap but distinct focuses. Generative AI and LLMs primarily focus on the development and ap"


In [8]:
summary_dir = ROOT / 'results_summary'
summary_dir.mkdir(exist_ok=True)
complex_df.to_csv(summary_dir / 'complex_compare_details.csv', index=False)
pivot_answers.to_csv(summary_dir / 'complex_compare_pivot.csv')
summary_df.to_csv(summary_dir / 'complex_compare_summary.csv', index=False)
print(f'Saved complex comparison CSV files to: {summary_dir.resolve()}')


Saved complex comparison CSV files to: /Users/Lenovo/Desktop/sem 6/Gen_AI group assignment/assignment 1/results_summary
